# Chapter 10 — VirtIO: Transport, Queues, and Device Types

## Concept
VirtIO (OASIS spec) defines a paravirtual I/O framework. Components: the
virtqueue (descriptor ring, available ring, used ring), split-ring vs packed-ring
layout, MMIO vs PCI transport, feature negotiation (FEATURES bits).
Device types: virtio-blk (block), virtio-net (network), virtio-scsi, virtio-rng.
QEMU QMP exposes `device_add` / `device_del` for hot-plug.

## Lab Objectives
1. Hot-plug a `virtio-blk` device via QMP.
2. Confirm the device appears in `/sys/bus/virtio/devices/` and `/dev/`.
3. Perform a `dd` read from the device — confirm I/O completes without error.
4. Hot-remove the device via QMP — confirm clean detach.

> **QEMU Fidelity Note** — Results from this lab are functionally valid on
> QEMU `virt` machine with HVF (macOS Apple Silicon). Behaviours that differ
> from real Neoverse silicon are annotated inline. See Section 6 of the
> project plan for the full gap table.


In [ ]:
import sys, os, pathlib, time
from IPython.display import display, HTML

# ── USER CONFIGURATION — edit these paths before running ──────────────────────
LABS_ROOT    = pathlib.Path.home() / "arm_qemu_labs"
SHARED_DIR   = LABS_ROOT / "shared"
DISK_IMAGE   = LABS_ROOT / "images" / "ubuntu-24.04-arm64.qcow2"
SEED_ISO     = LABS_ROOT / "images" / "seed.iso"   # cloud-init seed
FIRMWARE     = LABS_ROOT / "firmware" / "QEMU_EFI.fd"
CONSOLE_USER = "ubuntu"
CONSOLE_PASS = "arm-lab-2026"
CPU          = "cortex-a76"
RAM          = "2G"
SMP          = 1
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(SHARED_DIR))
from qemu_launcher  import QEMULauncher
from qmp_client     import QMPClient
from serial_console import SerialConsole
from assert_lib     import (assert_true, assert_false, assert_equal,
                             assert_contains, assert_not_contains,
                             assert_qmp_ok, assert_in_range, summary, reset)
reset()

import shutil
assert shutil.which("qemu-system-aarch64"), (
    "FATAL: qemu-system-aarch64 not found in PATH. Run setup_qemu_labs.sh.")
assert FIRMWARE.exists(),   f"FATAL: Firmware not found at {FIRMWARE}"
assert DISK_IMAGE.exists(), f"FATAL: Disk image not found at {DISK_IMAGE}"
assert SEED_ISO.exists(),   f"FATAL: Seed ISO not found at {SEED_ISO}"
print("✓ Environment check passed")
print(f"  qemu-system-aarch64 : {shutil.which('qemu-system-aarch64')}")
print(f"  Firmware            : {FIRMWARE}")
print(f"  Disk image          : {DISK_IMAGE}")


In [ ]:
launcher = QEMULauncher(
    cpu=CPU, ram=RAM, smp=SMP,
    firmware=str(FIRMWARE),
    disk_image=str(DISK_IMAGE),
    seed_iso=str(SEED_ISO),
    extra_args=["-netdev", "user,id=net0", "-device", "virtio-net-pci,netdev=net0"],
)
launcher.launch()
launcher.wait_ready(timeout=15)
print(f"QEMU running — QMP port {launcher.qmp_port}, serial port {launcher.serial_port}")


In [ ]:
qmp = QMPClient(port=launcher.qmp_port)
qmp.connect(timeout=30)

sc = SerialConsole(port=launcher.serial_port)
sc.connect(timeout=30)

print("Waiting for guest boot (up to 120 s on HVF, longer on TCG)…")
sc.wait_for_boot(timeout=180)
sc.login(CONSOLE_USER, CONSOLE_PASS)
print("Guest ready.")


In [ ]:
# ── Step 1: List pre-existing virtio devices ─────────────────────────────────
virtio_pre = sc.run_command(
    "ls /sys/bus/virtio/devices/ 2>/dev/null || echo 'none'", timeout=10
)
print("Pre-existing virtio devices:", virtio_pre)

dev_pre = sc.run_command("ls /dev/vd* 2>/dev/null || echo 'none'", timeout=10)
print("Pre-existing /dev/vd* block devices:", dev_pre)


In [ ]:
# ── Step 2: Create a test disk image (on host) for hot-plug ──────────────────
import subprocess, os, tempfile

hotplug_img = os.path.expanduser("~/arm_qemu_labs/images/hotplug_test.qcow2")
if not os.path.exists(hotplug_img):
    result = subprocess.run(
        ["qemu-img", "create", "-f", "qcow2", hotplug_img, "256M"],
        capture_output=True, text=True
    )
    print("qemu-img create:", result.stdout + result.stderr)
else:
    print(f"Hot-plug image already exists: {hotplug_img}")


In [ ]:
# ── Step 3: Hot-plug virtio-blk via QMP device_add ───────────────────────────
import json

# Add the disk backend as a blockdev first
blockdev_resp = qmp.send_command("blockdev-add", {
    "driver": "qcow2",
    "node-name": "hotdisk0",
    "file": {
        "driver": "file",
        "filename": hotplug_img,
    },
})
print("blockdev-add:", json.dumps(blockdev_resp, indent=2))

# Attach as a virtio-blk device
virtblk_resp = qmp.device_add(
    driver="virtio-blk-pci",
    device_id="hotblk0",
    drive="hotdisk0",
)
print("device_add virtio-blk-pci:", json.dumps(virtblk_resp, indent=2))


In [ ]:
# ── Step 4: Wait for guest kernel to enumerate the new device ────────────────
time.sleep(3)

virtio_post = sc.run_command(
    "ls /sys/bus/virtio/devices/ 2>/dev/null", timeout=10
)
print("virtio devices after hot-plug:", virtio_post)

dev_post = sc.run_command("ls /dev/vd* 2>/dev/null || echo 'none'", timeout=10)
print("/dev/vd* after hot-plug:", dev_post)


In [ ]:
# ── Step 5: Read from the hot-plugged device ──────────────────────────────────
# Find the new block device name
new_devs = [d for d in dev_post.split() if d.startswith("/dev/vd")
            and d not in dev_pre.split()]
if new_devs:
    test_dev = new_devs[0]
    print(f"Testing read from {test_dev} …")
    dd_out = sc.run_command(
        f"sudo dd if={test_dev} of=/dev/null bs=4096 count=64 2>&1",
        timeout=30
    )
    print("dd output:", dd_out)
else:
    print("No new /dev/vd* device found after hot-plug")
    test_dev = None
    dd_out = "SKIP"


In [ ]:
# ── Step 6: Hot-remove the device via QMP ────────────────────────────────────
del_resp = qmp.device_del("hotblk0")
print("device_del:", json.dumps(del_resp, indent=2))
time.sleep(2)

dev_after_del = sc.run_command("ls /dev/vd* 2>/dev/null || echo 'none'", timeout=10)
print("/dev/vd* after hot-remove:", dev_after_del)


In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert_true("hotdisk0" not in str(blockdev_resp.get("error", "")),
            "blockdev-add succeeded",
            detail=str(blockdev_resp)[:100],
            action="Check hot-plug image path and qemu-img format")

virtio_post_devices = virtio_post.strip().split()
assert_true(len(virtio_post_devices) > 0,
            "virtio devices present in /sys/bus/virtio/devices/",
            detail=str(virtio_post_devices[:5]),
            action="Hot-plug may have failed; check QMP device_add response")

if test_dev:
    assert_contains(dd_out, r"\d+ bytes|records in|records out",
                    f"dd read from {test_dev} completes",
                    detail=dd_out[:100],
                    action="Verify device node permissions and virtio-blk driver")
else:
    assert_true(False,
                "New /dev/vd* device created after hot-plug",
                action="Guest kernel did not create block device node — check virtio-blk driver")

# Hot-remove: device should no longer appear
assert_not_contains(dev_after_del, test_dev or "vdb",
                    "Hot-removed device absent from /dev/ after device_del",
                    action="Guest kernel still holds device reference; check udev rules")


In [ ]:
# ── Teardown: always runs even if earlier cells raised ────────────────────────
try:
    qmp.quit()
except Exception:
    pass
try:
    sc.close()
except Exception:
    pass
try:
    launcher.terminate()
except Exception:
    pass
print("Teardown complete. QEMU process terminated.")


## Lab Summary

| Assertion | Expected | Notes |
|-----------|----------|-------|
| blockdev-add succeeds | Yes | qcow2 backend attached |
| virtio device in sysfs | Present | Driver probed |
| /dev/vd* node created | Yes | Block device enumerated |
| dd read completes | Yes | I/O path functional |
| device absent after del | Yes | Clean hot-remove |

> VirtIO hot-plug is fully functional in QEMU. The virtqueue descriptor ring
> layout and feature bit negotiation are identical to production firmware paths.

## Next Steps
→ **Chapter 11**: PCIe on ARM — enumerate the PCIe bus and verify MSI-X routing via ITS.
